
# Laboratório 3 — Alinhamento, Homografia 2D e Mosaico

**Disciplina:** ESZA019 — Visão Computacional  
**Docente:** Prof. Celso Setsuo Kurashima  
**Autores:** Cesar de Jesus; Mariana Chiara; Vinicius de Marchi  
**Data de realização dos experimentos:** 17/06/2026  
**Data de publicação do relatório:** 28/06/2026  

---

## Resumo

Este relatório apresenta a implementação e a análise de um pipeline de alinhamento e costura de imagens baseado em características locais. Foram capturados pares de imagens com duas webcams em dois cenários: uma pessoa e a capa plana de um livro. Em cada experimento, pontos de interesse e descritores foram obtidos com SIFT, as correspondências foram filtradas pelo teste de razão de Lowe e duas matrizes de homografia foram estimadas: uma por mínimos quadrados tradicionais e outra de forma robusta com RANSAC. Os resultados demonstraram a elevada sensibilidade do ajuste por mínimos quadrados à presença de correspondências incorretas. O RANSAC produziu alinhamentos geometricamente mais coerentes, apesar de permanecer limitado por paralaxe, movimento da pessoa, padrões repetitivos e diferenças entre as câmeras. Também foi implementada uma versão em tempo real, com leitura simultânea das webcams, atualização das correspondências e geração contínua do mosaico.



## 1. Introdução


O alinhamento de imagens procura determinar uma transformação geométrica capaz de relacionar pontos correspondentes observados em duas ou mais imagens, operação central em aplicações como panoramas, mosaicos, registro médico, sensoriamento remoto, inspeção industrial, realidade aumentada, localização visual e mapeamento robótico.

Em um pipeline baseado em características, o processamento é dividido em quatro etapas principais: detecção de pontos de interesse, descrição das vizinhanças, correspondência dos descritores e estimação de uma transformação geométrica. Para cenas aproximadamente planas ou para imagens capturadas por rotação em torno de um mesmo centro óptico, a relação projetiva entre duas vistas pode ser descrita por uma homografia bidimensional. Entretanto, os pares inicialmente correspondidos podem conter *outliers*, associações visualmente parecidas mas geometricamente incorretas, e o método de Mínimos Quadrados pode ser fortemente distorcido por esse erros, uma vez que utiliza todas as correspondências. O RANSAC reduz esse problema ao estimar repetidamente o modelo a partir de amostras mínimas e selecionar a hipótese apoiada pelo maior conjunto consistente de pontos.

Neste trabalho, o pipeline foi aplicado a dois pares de imagens estáticas e posteriormente adaptado para a leitura contínua de duas webcams, consolidando os conceitos vistos em aulas teóricas da disciplina.


## 2. Objetivos

O laboratório teve como objetivos:

1. Detectar e descrever características locais com SIFT;
2. Realizar o casamento de descritores e eliminar correspondências ambíguas com o teste de Lowe;
3. Estimar homografias por mínimos quadrados e por RANSAC;
4. Quantificar inliers, outliers e diferenças entre as matrizes estimadas;
5. Produzir mosaicos a partir das imagens alinhadas;
6. Analisar limitações geométricas e artefatos visuais;
7. Adaptar o processamento para duas webcams em tempo real;
8. Relacionar alinhamento e costura com o trabalho final de detecção de postura.

## 3. Fundamentação Teórica

### 3.1 Características locais

Uma característica local é uma região da imagem suficientemente distintiva para ser detectada novamente em outra vista. O pipeline usual possui três componentes:

- **Detecção:** localização dos pontos de interesse;
- **Descrição:** representação numérica da vizinhança de cada ponto;
- **Matching:** busca de descritores semelhantes entre duas imagens.

Características adequadas devem apresentar repetibilidade diante de alterações moderadas de posição, escala, rotação, iluminação e ponto de vista.

### 3.2 SIFT

O ***Scale-Invariant Feature Transform* (SIFT)** detecta extremos em uma representação multiescala da imagem e atribui a cada ponto uma escala e uma orientação dominantes. O descritor é construído a partir de histogramas locais de orientação de gradientes, resultando em um vetor de 128 componentes.

Essa construção torna o método robusto a mudanças de escala e rotação e relativamente resistente a alterações fotométricas e pequenas transformações afins. No OpenCV, a detecção e a descrição são realizadas por:

```python
    sift = cv2.SIFT_create()
    pontos, descritores = sift.detectAndCompute(imagem_cinza, None)
```


### 3.3 Correspondência e Teste de Razão de Lowe

Os descritores **SIFT** são comparados utilizando a **distância Euclidiana**. O `BFMatcher`, configurado para buscar os dois vizinhos mais próximos (*k = 2*), retorna, para cada descritor da primeira imagem, as duas melhores correspondências na segunda imagem.

O **teste da razão de Lowe** aceita uma correspondência quando:

$$
d_1 < \tau \, d_2
$$

em que:

- $d_1$ é a distância até o vizinho mais próximo;
- $d_2$ é a distância até o segundo vizinho mais próximo;
- $\tau = 0{,}75$ é o limiar utilizado neste laboratório.

A ideia é rejeitar correspondências ambíguas, isto é, casos em que as duas melhores opções possuem distâncias muito semelhantes. Quando $d_1$ é significativamente menor que $d_2$, a correspondência é considerada mais confiável.

### 3.4 Homografia 2D

A homografia é uma transformação projetiva representada por uma matriz $3 \times 3$:

$$
\mathbf{H} =
\begin{bmatrix}
h_{00} & h_{01} & h_{02} \\
h_{10} & h_{11} & h_{12} \\
h_{20} & h_{21} & h_{22}
\end{bmatrix}.
$$

Em coordenadas homogêneas, a relação entre um ponto da primeira imagem $(x, y)$ e seu correspondente na segunda imagem $(x', y')$ é dada por:

$$
\lambda
\begin{bmatrix}
x' \\
y' \\
1
\end{bmatrix}
=
\mathbf{H}
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix},
$$

em que $\lambda$ é um fator de escala.

Após a multiplicação, as coordenadas cartesianas são obtidas por:

$$
x' =
\frac{h_{00}x + h_{01}y + h_{02}}
     {h_{20}x + h_{21}y + h_{22}},
$$

$$
y' =
\frac{h_{10}x + h_{11}y + h_{12}}
     {h_{20}x + h_{21}y + h_{22}}.
$$

Como matrizes que diferem apenas por um fator escalar representam a mesma transformação, a homografia possui oito graus de liberdade. Por convenção, normaliza-se o elemento $h_{22} = 1$, desde que ele seja diferente de zero.

Uma única homografia modela corretamente a relação entre duas imagens quando a cena é aproximadamente plana ou quando a câmera realiza apenas uma rotação em torno do seu centro óptico. Em cenas tridimensionais, deslocamentos da câmera ou objetos em movimento geram paralaxe, violando a hipótese de uma transformação projetiva global.

### 3.5 Mínimos Quadrados

Com mais correspondências do que o mínimo necessário, obtém-se um sistema sobredeterminado e o método de mínimos quadrados procura a matriz que minimiza a soma dos erros de reprojeção. Contudo, o erro quadrático atribui grande influência a correspondências incorretas e, assim, poucos outliers podem alterar fortemente a homografia.

### 3.6 RANSAC

O **Random Sample Consensus (RANSAC)** estima modelos robustos na presença de erros grosseiros:

1. seleciona aleatoriamente uma amostra mínima;
2. estima uma homografia;
3. reprojeta todos os pontos;
4. classifica como *inliers* os pontos cujo erro fica abaixo de um limiar;
5. repete o processo e mantém o modelo com maior consenso;
6. refina a estimativa com os inliers selecionados.

Neste trabalho foi usado limiar de reprojeção de 5 pixels.

### 3.7 Warping, mosaico e blending

Depois da estimação de \(\mathbf{H}\), uma imagem é reprojetada para o sistema de coordenadas da outra por `cv2.warpPerspective`. O mosaico básico utilizado no experimento cria um *canvas* comum, insere a imagem transformada e copia a segunda imagem para sua posição.

Essa composição direta não realiza suavização da sobreposição. Portanto, diferenças de exposição, balanço de branco, paralaxe e pequenos erros geométricos podem gerar linhas de corte e *ghosting*. Técnicas como compensação de ganho, *feathering* e *multi-band blending* são usadas para suavizar essas transições.


## 4. Procedimentos Experimentais

### 4.1 Materiais e Ambiente

- Duas webcams;
- Computador com Ubuntu Linux;
- Python 3;
- OpenCV;
- NumPy;
- Matplotlib;
- Ambiente virtual Conda.

As imagens foram adquiridas com resolução de \(640\times480\) pixels.

### 4.2 Captura Simultânea

O primeiro programa abriu as câmeras de índices 0 e 1 simultaneamente. A tecla `S` salvou os quadros como `imagem1.jpg` e `imagem2.jpg`, enquanto `Q` encerrou a captura. As câmeras foram posicionadas de modo a manter mais de 50% de sobreposição entre as vistas.

### 4.3 Processamento Estático

O processamento foi executado separadamente para:

- **Experimento A.1:** pessoa;
- **Experimento A.2:** livro.

O procedimento foi:

1. Leitura das imagens;
2. Conversão para escala de cinza;
3. Detecção e descrição SIFT;
4. Correspondência com `BFMatcher`;
5. Teste de Lowe com $\tau = 0{,}75$;
6. Estimação de $\mathbf{H}_{LS}$ com todas as correspondências;
7. Estimação de $\mathbf{H}_{RANSAC}$ com limiar de 5 pixels;
8. Contagem de inliers e outliers;
9. Criação de mosaicos com `warpPerspective`;
10. Salvamento das imagens de correspondências e dos mosaicos.

### 4.4 Processamento em Tempo Real

O terceiro programa abriu as duas webcams, repetiu o pipeline SIFT–matching–RANSAC e exibiu em uma janela:

- correspondências aceitas pelo RANSAC;
- mosaico atualizado;
- número de pontos SIFT, matches, inliers e outliers.

Para reduzir o custo computacional, o processamento completo foi realizado a cada três quadros. A tecla `S` salvou os resultados correntes e `Q` encerrou a execução.


## 5. Resultados e Análise

### 5.1 Resumo Quantitativo

| Experimento | Pontos SIFT (img. 1) | Pontos SIFT (img. 2) | Bons matches | Inliers | Outliers | Outliers (%) | abs(Δh₀₂) (px) | abs(Δh₁₂) (px) |
|:------------|---------------------:|---------------------:|-------------:|---------:|----------:|-------------:|----------------:|----------------:|
| Pessoa | 735 | 660 | 67 | 38 | 29 | 43.28 | 10.11 | 270.69 |
| Livro  | 978 | 951 | 104 | 23 | 81 | 77.88 | 267.82 | 609.49 |

A porcentagem de *outliers* foi calculada por:

$$
\%\,\text{outliers}
=
\left(
1-
\frac{N_{\text{inliers}}}{N_{\text{matches}}}
\right)\times100.
$$

### 5.2 Experimento A.1 - Pessoa

#### Imagens de entrada

<table>
<tr>
<td style="text-align:center"><img src="assets/pessoa_imagem1.jpg" width="430"><br><b>Imagem 1</b></td>
<td style="text-align:center"><img src="assets/pessoa_imagem2.jpg" width="430"><br><b>Imagem 2</b></td>
</tr>
</table>

O SIFT detectou **735** pontos na primeira imagem e **660** na segunda. Após o teste de Lowe, permaneceram **67** correspondências. O RANSAC classificou **38** como inliers e **29** como outliers:

$$
\%\,\text{outliers}
=
\left(
1 -
\frac{38}{67}
\right)
\times 100
=
43.28\%.
$$

#### Correspondências aceitas pelo RANSAC

![Correspondências da pessoa](assets/pessoa_correspondencias_ransac.jpg)

As correspondências concentram-se principalmente no moletom, em suas letras, cordões, dobras e bordas. A elevada quantidade de outliers pode ser explicada por:

- mudança de enquadramento entre rosto e tronco;
- profundidade tridimensional do corpo;
- pequenas mudanças de postura;
- dobras repetitivas do tecido;
- variações de iluminação e exposição entre as webcams;
- fundo com estruturas repetidas.

Uma pessoa não é uma superfície plana e pode se mover entre as capturas. Logo, uma única homografia não consegue representar perfeitamente toda a cena.


#### Matrizes estimadas

**Mínimos quadrados**

```text
[-0.531139   0.847119   39.396703]
[-0.694630   0.589054   150.211227]
[-0.003707   0.002596   1.000000]
```

**RANSAC**

```text
[ 0.810098   0.038887   29.284945]
[-0.168393   0.863849  -120.482857]
[-0.000322   0.000025   1.000000]
```

As diferenças absolutas entre os termos de translação foram

$$
|\Delta h_{02}|
=
\left|39.396703 - 29.284945\right|
=
10.111758\ \text{pixels}.
$$

$$
|\Delta h_{12}|
=
\left|150.211227 - (-120.482857)\right|
=
270.694084\ \text{pixels}.
$$

A discrepância vertical é especialmente elevada. O ajuste por mínimos quadrados tentou reduzir simultaneamente os erros de todas as correspondências, inclusive as incorretas, resultando em uma transformação fisicamente incompatível com a vista predominante.


#### Comparação dos mosaicos

<table>
<tr>
<td style="text-align:center"><img src="assets/pessoa_mosaico_minimos_quadrados.jpg" width="430"><br><b>Mínimos quadrados</b></td>
<td style="text-align:center"><img src="assets/pessoa_mosaico_ransac.jpg" width="430"><br><b>RANSAC</b></td>
</tr>
</table>

O mosaico por mínimos quadrados apresenta deformação projetiva extrema, com grande parte da imagem comprimida em forma de leque. O resultado com RANSAC preserva melhor a geometria da pessoa e do fundo, embora ainda apresente bordas pretas, corte abrupto e limitações decorrentes da não planaridade do corpo.

### 5.3 Experimento A.2 - Livro

#### Imagens de entrada

<table>
<tr>
<td style="text-align:center"><img src="assets/livro_imagem1.jpg" width="430"><br><b>Imagem 1</b></td>
<td style="text-align:center"><img src="assets/livro_imagem2.jpg" width="430"><br><b>Imagem 2</b></td>
</tr>
</table>

O SIFT detectou **978** pontos na primeira imagem e **951** na segunda. Foram mantidas **104** correspondências após Lowe. O RANSAC aceitou **23** inliers e rejeitou **81** outliers:

$$
\%\,\text{outliers}
=
\left(
1 -
\frac{23}{104}
\right)
\times 100
=
77.88\%.
$$

#### Correspondências aceitas pelo RANSAC

![Correspondências do livro](assets/livro_correspondencias_ransac.jpg)

Embora a capa seja aproximadamente plana e, portanto, adequada a uma homografia, o número de outliers foi maior do que no experimento com a pessoa. A capa possui padrões visualmente repetitivos: círculos amarelos, quadrados amarelos, linhas vermelhas e segmentos azuis. Descritores de diferentes elementos podem ser semelhantes, gerando correspondências falsas. Também houve diferença de enquadramento, rotação, escala aparente e exposição entre as webcams.


#### Matrizes estimadas

**Mínimos quadrados**

```text
[-0.085984  -0.828821   262.253080]
[-0.108816  -0.827109   273.669301]
[-0.000335  -0.003122     1.000000]
```

**RANSAC**

```text
[ 1.402489  -0.028172    -5.567021]
[ 0.060191   1.334126  -335.819808]
[ 0.000103   0.000011     1.000000]
```

As diferenças absolutas entre as componentes de translação são:

$$
|\Delta h_{02}|
=
\left|262.253080 - (-5.567021)\right|
=
267.820101\ \text{pixels}.
$$

$$
|\Delta h_{12}|
=
\left|273.669301 - (-335.819808)\right|
=
609.489109\ \text{pixels}.
$$

A diferença elevada confirma que as falsas correspondências dominaram o ajuste tradicional. O RANSAC identificou um subconjunto menor, porém geometricamente consistente, suficiente para recuperar a transformação do plano da capa.


#### Comparação dos mosaicos

<table>
<tr>
<td style="text-align:center"><img src="assets/livro_mosaico_minimos_quadrados.jpg" width="430"><br><b>Mínimos quadrados</b></td>
<td style="text-align:center"><img src="assets/livro_mosaico_ransac.jpg" width="430"><br><b>RANSAC</b></td>
</tr>
</table>

O resultado por RANSAC preserva a estrutura retangular da capa e combina regiões complementares das duas vistas. Já o mosaico obtido com todas as correspondências não representa adequadamente a transformação entre as imagens. Esse experimento mostra que uma cena plana não garante, por si só, um bom resultado: ainda é necessário rejeitar correspondências incorretas.


### 5.4 Parte B - Processamento em Tempo Real

O programa em tempo real atualizou continuamente os descritores, as correspondências, a homografia e o mosaico. O quadro abaixo, extraído da gravação, mostra o livro observado simultaneamente pelas duas webcams.

![Execução em tempo real](assets/parte_b_tempo_real_frame.png)

No quadro ilustrado foram mostrados aproximadamente:

- SIFT 1: 1024 pontos;
- SIFT 2: 1041 pontos;
- Bons matches: 154;
- Inliers: 67;
- Outliers: 87;
- Proporção instantânea de outliers: \(87/154 \times 100 \approx 56{,}49\%\).

Os valores variaram ao longo do vídeo, pois cada quadro contém pequenas alterações de foco, ruído, movimento e iluminação. O processamento a cada três quadros tornou a execução mais viável, mas também reduziu a taxa de atualização.

<video width="720" controls>
  <source src="assets/parte_b_tempo_real.webm" type="video/webm">
  O navegador não conseguiu reproduzir o vídeo.
</video>

[**Abrir ou baixar a gravação da Parte B**](assets/parte_b_tempo_real.webm)


## 6. Questões de Análise

### 6.1 Por que $h_{22}$ é normalizado como 1?

A homografia opera em coordenadas homogêneas e é definida apenas a menos de um fator escalar. Assim, $\mathbf{H}$ e $k\mathbf{H}$, para $k \neq 0$, representam o mesmo mapeamento após a divisão pela coordenada homogênea. Fixar $h_{22}=1$ remove essa ambiguidade de escala e facilita a comparação numérica das matrizes.

### 6.2 Número mínimo de correspondências

Uma matriz $3 \times 3$ possui nove elementos, mas apenas oito graus de liberdade devido à escala arbitrária. Cada par de pontos fornece duas equações lineares independentes. Portanto,

$$
N_{\min}=\frac{8}{2}=4.
$$

Assim, quatro pares de pontos são o mínimo absoluto para estimar uma homografia.

Esses quatro pontos não podem formar uma configuração degenerada. Em particular, não devem ser todos colineares e precisam distribuir-se de modo a fornecer informação projetiva suficiente.

### 6.3 Superfície uniforme e navegação robótica

Em uma superfície sem textura, os gradientes locais são fracos e o SIFT detecta poucos ou nenhum ponto distintivo. Sem descritores, não existem correspondências suficientes; sem pelo menos quatro pares consistentes, a homografia não pode ser estimada.

Em um robô ou drone, isso provoca perda de localização visual, instabilidade da odometria e impossibilidade de registrar novos quadros no mapa. Uma aplicação real deve combinar visão com outros sensores, utilizar marcadores artificiais ou mudar o ponto de vista até encontrar regiões texturizadas.

### 6.4 Localização da emenda e artefatos

No código implementado, a segunda imagem é copiada diretamente sobre o *canvas* após a transformação da primeira. Por isso, a emenda coincide com o contorno retangular da segunda imagem dentro do mosaico.

Nos resultados observam-se:

- mudanças abruptas de luminosidade;
- regiões pretas sem informação;
- pequenas descontinuidades nas bordas;
- deformações quando o modelo é inadequado;
- possibilidade de *ghosting* em regiões de movimento ou paralaxe.

No experimento com a pessoa, o corpo e o fundo estão em profundidades diferentes, e pequenas mudanças de postura impedem que uma transformação global alinhe todos os pixels. No experimento com o livro, a emenda é geometricamente mais consistente, mas diferenças fotométricas entre as webcams permanecem visíveis.

### 6.5 Técnica de pós-processamento

Uma melhoria apropriada é o **multi-band blending**, combinado com compensação de ganho. A ideia é decompor cada imagem em bandas de frequência por meio de pirâmides Laplacianas. Em cada nível $\ell$, as imagens são combinadas com uma máscara suave $W_\ell$:

$$
L_\ell =
W_\ell\,L_\ell(I_1)
+
\left(1-W_\ell\right)L_\ell(I_2).
$$

A imagem final é reconstruída pela soma das bandas. As transições de baixa frequência, como iluminação e cor, são misturadas em uma área ampla, enquanto detalhes de alta frequência, como bordas e texturas, são combinados em uma faixa menor. Isso reduz linhas de corte sem apagar excessivamente os detalhes.

Antes do *blending*, uma compensação de ganho pode aproximar as intensidades médias na região de sobreposição:

$$
I'_2 = gI_2 + b,
$$

em que $g$ representa o ganho e $b$ o deslocamento (*bias*), estimados a partir dos pixels correspondentes. Em uma implementação mais simples e rápida, o *feathering* utiliza pesos que variam gradualmente com a distância à borda, mas o *multi-band blending* tende a produzir resultados superiores quando as diferenças de exposição são relevantes.

## 7. Aplciação no Trabalho Final - Detecção de Postura

O trabalho final do grupo consiste em um sistema de detecção de postura com webcam e geração de alertas. As técnicas deste laboratório podem contribuir de diferentes formas:

1. **Registro de duas câmeras:** uma homografia calculada sobre partes estáticas e aproximadamente planas do ambiente pode colocar as duas vistas em um referencial comum;
2. **Ampliação do campo de visão:** a costura pode reduzir pontos cegos e manter a pessoa visível quando ela se desloca;
3. **Normalização do plano de trabalho:** mesas, paredes, telas ou o chão podem ser retificados para facilitar a definição de regiões de interesse;
4. **Fusão de informações:** pontos detectados em diferentes vistas podem ser associados após a calibração geométrica;
5. **Compensação de pequenos deslocamentos da câmera:** o alinhamento do fundo pode estabilizar a região usada pelo detector.

Entretanto, a homografia **não deve ser usada para deformar o corpo inteiro como se ele fosse plano**. O experimento com a pessoa demonstrou que corpo, cadeira, parede e demais elementos estão em profundidades diferentes. Para estimar postura em múltiplas câmeras, o procedimento mais correto é calibrar as câmeras, obter parâmetros intrínsecos e extrínsecos e, quando necessário, triangular pontos corporais em 3D. A homografia permanece útil para planos específicos, como o chão, ou para estabilizar o fundo.

Uma arquitetura prática para o projeto final seria:

1. Detectar pontos corporais em cada câmera;
2. Utilizar uma região estática do cenário para estimar ou verificar o alinhamento;
3. Mapear pontos de contato, como os pés, para o plano do chão;
4. Fundir as detecções quando ambas as câmeras observarem a mesma pessoa;
5. Calcular ângulos posturais e disparar alertas;
6. Manter a última transformação válida e aplicar suavização temporal para reduzir oscilações.

O processamento em tempo real também revelou a necessidade de otimização. Estratégias futuras incluem limitar o SIFT a uma região estática, rastrear inliers com fluxo óptico, estimar a homografia apenas quando necessário e suavizar seus parâmetros ao longo do tempo.


## 8. Limitações e Melhorias Propostas

As principais limitações observadas foram:

- Câmeras diferentes, com respostas fotométricas e parâmetros internos distintos;
- Ausência de calibração e correção de distorção;
- Movimento e não planaridade no experimento com a pessoa;
- Padrões repetitivos na capa do livro;
- Composição por sobrescrita, sem blending;
- Recomputação independente da homografia no vídeo, causando possíveis oscilações;
- Bordas pretas resultantes do warping.

Como melhorias, propõem-se:

- Calibração individual das webcams e correção da distorção radial;
- Sincronização mais precisa das capturas;
- Uso de verificação cruzada ou correspondência bidirecional;
- Ajuste adaptativo do limiar de Lowe;
- Uso de PROSAC, MAGSAC ou RANSAC com critérios adicionais;
- Compensação de exposição;
- Seleção automática da melhor linha de costura;
- Multi-band blending;
- Suavização temporal da homografia;
- Detecção de falhas e reaproveitamento da última transformação válida.


## 9. Conclusões

O laboratório permitiu implementar todas as etapas básicas de um sistema de alinhamento e costura baseado em características. O SIFT forneceu pontos e descritores repetíveis, enquanto o teste de Lowe reduziu correspondências ambíguas. Entretanto, uma fração significativa de falsos matches permaneceu nos dois experimentos.

A comparação entre os métodos demonstrou que mínimos quadrados tradicionais são inadequados quando existem outliers. No experimento com a pessoa, 43,28% das correspondências filtradas foram classificadas como outliers. No livro, padrões repetitivos elevaram essa proporção para 77,88%. Em ambos os casos, o RANSAC recuperou um modelo mais coerente e produziu mosaicos visualmente superiores.

O experimento com a pessoa também mostrou uma limitação fundamental: uma homografia é um modelo global apropriado para planos ou rotações com centro óptico aproximadamente fixo, não para toda uma cena tridimensional e deformável. O livro apresentou condições geométricas mais adequadas, mas ainda exigiu estimação robusta por causa das repetições visuais.

A versão em tempo real comprovou a possibilidade de executar o pipeline continuamente com duas webcams. Contudo, o custo do SIFT e a variabilidade temporal indicam a necessidade de otimizações, rastreamento e suavização para uma aplicação final estável.

Por fim, o alinhamento e a homografia podem apoiar o projeto de detecção de postura na estabilização do cenário, no mapeamento de planos e na combinação de campos de visão, desde que não sejam usados como substitutos da calibração multivista e da modelagem tridimensional do corpo.



## 10. Referências

BROWN, M.; LOWE, D. G. Automatic panoramic image stitching using invariant features. *International Journal of Computer Vision*, v. 74, n. 1, p. 59–73, 2007. DOI: 10.1007/s11263-006-0002-3.

FISCHLER, M. A.; BOLLES, R. C. Random sample consensus: a paradigm for model fitting with applications to image analysis and automated cartography. *Communications of the ACM*, v. 24, n. 6, p. 381–395, 1981. DOI: 10.1145/358669.358692.

KURASHIMA, C. S. *ESZA019 — Visão Computacional: Descritores e Matching, Transformações Geométricas, Alinhamento e Panoramas.*. UFABC, 2026.

KURASHIMA, C. S. *ESZA019 — Visão Computacional: Laboratório 3 — Alinhamento, Homografia 2D e Mosaico*. UFABC, 2026.

LOWE, D. G. Distinctive image features from scale-invariant keypoints. *International Journal of Computer Vision*, v. 60, n. 2, p. 91–110, 2004. DOI: 10.1023/B:VISI.0000029664.99615.94.

SZELISKI, R. *Computer Vision: Algorithms and Applications*. 2. ed. Cham: Springer, 2022. DOI: 10.1007/978-3-030-34372-9.

SZELISKI, R. Image alignment and stitching: a tutorial. *Foundations and Trends in Computer Graphics and Vision*, v. 2, n. 1, p. 1–104, 2006.

OPENCV. Feature Matching + Homography to find Objects. OpenCV-Python Tutorials. Disponível em: <https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html>. Acesso em: 27 jun. 2026.


---

# Apêndice A — Código de captura simultânea

Arquivo correspondente: [`codigos/Lab3_captura_duas_webcams.py`](codigos/Lab3_captura_duas_webcams.py)


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# ============================================================
# LAB 3 - CAPTURA SIMULTÂNEA COM DUAS WEBCAMS
#
# Teclas:
#   S -> salva imagem1.jpg e imagem2.jpg
#   Q -> encerra o programa
#
# Caso as câmeras não abram, troque os índices abaixo.
# Exemplos comuns:
#   câmera integrada = 0
#   câmera USB = 1 ou 2
# ============================================================

CAMERA_1 = 0
CAMERA_2 = 1

NOME_IMAGEM_1 = "imagem1.jpg"
NOME_IMAGEM_2 = "imagem2.jpg"

LARGURA = 640
ALTURA = 480


def abrir_camera(indice):
    """
    Tenta abrir uma câmera pelo índice informado.
    No Windows, utiliza inicialmente o backend DirectShow.
    """
    cap = cv2.VideoCapture(indice, cv2.CAP_DSHOW)

    if not cap.isOpened():
        cap.release()
        cap = cv2.VideoCapture(indice)

    if cap.isOpened():
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, LARGURA)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTURA)

    return cap


def ajustar_altura(imagem, altura_alvo):
    """
    Redimensiona a imagem mantendo a proporção.
    """
    altura, largura = imagem.shape[:2]

    if altura == altura_alvo:
        return imagem

    nova_largura = int(largura * altura_alvo / altura)
    return cv2.resize(imagem, (nova_largura, altura_alvo))


def adicionar_texto(imagem, texto):
    """
    Adiciona uma legenda na parte superior da imagem.
    """
    resultado = imagem.copy()

    cv2.rectangle(
        resultado,
        (0, 0),
        (resultado.shape[1], 40),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        resultado,
        texto,
        (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    return resultado


def main():
    cap1 = abrir_camera(CAMERA_1)
    cap2 = abrir_camera(CAMERA_2)

    if not cap1.isOpened():
        print(f"Erro: não foi possível abrir a câmera {CAMERA_1}.")
        print("Tente trocar CAMERA_1 para outro índice, como 1 ou 2.")

    if not cap2.isOpened():
        print(f"Erro: não foi possível abrir a câmera {CAMERA_2}.")
        print("Tente trocar CAMERA_2 para outro índice, como 0, 2 ou 3.")

    if not cap1.isOpened() or not cap2.isOpened():
        cap1.release()
        cap2.release()
        cv2.destroyAllWindows()
        return

    print("Duas câmeras abertas com sucesso.")
    print("Pressione S para salvar as duas imagens.")
    print("Pressione Q para sair.")

    while True:
        sucesso1, frame1 = cap1.read()
        sucesso2, frame2 = cap2.read()

        if not sucesso1:
            print("Erro ao capturar imagem da câmera 1.")
            break

        if not sucesso2:
            print("Erro ao capturar imagem da câmera 2.")
            break

        frame1_exibicao = adicionar_texto(
            frame1,
            "Camera 1 - imagem1.jpg"
        )

        frame2_exibicao = adicionar_texto(
            frame2,
            "Camera 2 - imagem2.jpg"
        )

        altura_comum = min(
            frame1_exibicao.shape[0],
            frame2_exibicao.shape[0]
        )

        frame1_exibicao = ajustar_altura(
            frame1_exibicao,
            altura_comum
        )

        frame2_exibicao = ajustar_altura(
            frame2_exibicao,
            altura_comum
        )

        visualizacao = np.hstack(
            (frame1_exibicao, frame2_exibicao)
        )

        cv2.putText(
            visualizacao,
            "S: salvar as duas imagens | Q: sair",
            (10, visualizacao.shape[0] - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.imshow(
            "Lab 3 - Captura com duas webcams",
            visualizacao
        )

        tecla = cv2.waitKey(1) & 0xFF

        if tecla == ord("s"):
            caminho1 = Path(NOME_IMAGEM_1)
            caminho2 = Path(NOME_IMAGEM_2)

            salvou1 = cv2.imwrite(str(caminho1), frame1)
            salvou2 = cv2.imwrite(str(caminho2), frame2)

            if salvou1 and salvou2:
                horario = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

                print("\nImagens salvas com sucesso!")
                print(f"{NOME_IMAGEM_1}: {caminho1.resolve()}")
                print(f"{NOME_IMAGEM_2}: {caminho2.resolve()}")
                print(f"Horário da captura: {horario}")

                aviso = visualizacao.copy()

                cv2.rectangle(
                    aviso,
                    (0, 0),
                    (aviso.shape[1], 65),
                    (0, 120, 0),
                    -1
                )

                cv2.putText(
                    aviso,
                    "IMAGENS SALVAS COM SUCESSO",
                    (20, 43),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.0,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA
                )

                cv2.imshow(
                    "Lab 3 - Captura com duas webcams",
                    aviso
                )

                cv2.waitKey(1200)

            else:
                print("Erro ao salvar uma ou ambas as imagens.")

        elif tecla == ord("q"):
            print("Programa encerrado pelo usuário.")
            break

    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()



# Apêndice B — Código do processamento estático

Arquivo correspondente: [`codigos/Lab3A_sift_homografia_mosaico.py`](codigos/Lab3A_sift_homografia_mosaico.py)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# LAB 3 - SIFT, HOMOGRAFIA, RANSAC E MOSAICO
#
# Arquivos de entrada:
#   imagem1.jpg
#   imagem2.jpg
#
# Arquivos gerados:
#   correspondencias_ransac.jpg
#   mosaico_minimos_quadrados.jpg
#   mosaico_ransac.jpg
#
# Execute este arquivo na mesma pasta das imagens.
# ============================================================

IMAGEM_1 = "imagem1.jpg"
IMAGEM_2 = "imagem2.jpg"

LIMIAR_LOWE = 0.75
LIMIAR_REPROJECAO = 5.0
MINIMO_MATCHES = 4


def normalizar_homografia(H):
    """Normaliza H para que H[2, 2] seja igual a 1."""
    if H is None:
        return None

    if abs(H[2, 2]) < 1e-12:
        return H

    return H / H[2, 2]


def criar_mosaico(imagem1, imagem2, H):
    """
    Projeta a imagem1 no plano da imagem2 usando H.
    O canvas é calculado para evitar cortes.
    """
    altura1, largura1 = imagem1.shape[:2]
    altura2, largura2 = imagem2.shape[:2]

    cantos1 = np.float32([
        [0, 0],
        [largura1, 0],
        [largura1, altura1],
        [0, altura1]
    ]).reshape(-1, 1, 2)

    cantos2 = np.float32([
        [0, 0],
        [largura2, 0],
        [largura2, altura2],
        [0, altura2]
    ]).reshape(-1, 1, 2)

    cantos1_transformados = cv2.perspectiveTransform(cantos1, H)
    todos_cantos = np.concatenate(
        (cantos1_transformados, cantos2),
        axis=0
    )

    x_min, y_min = np.floor(
        todos_cantos.min(axis=0).ravel()
    ).astype(int)

    x_max, y_max = np.ceil(
        todos_cantos.max(axis=0).ravel()
    ).astype(int)

    deslocamento_x = -x_min
    deslocamento_y = -y_min

    matriz_deslocamento = np.array([
        [1, 0, deslocamento_x],
        [0, 1, deslocamento_y],
        [0, 0, 1]
    ], dtype=np.float64)

    largura_canvas = x_max - x_min
    altura_canvas = y_max - y_min

    mosaico = cv2.warpPerspective(
        imagem1,
        matriz_deslocamento @ H,
        (largura_canvas, altura_canvas)
    )

    mosaico[
        deslocamento_y:deslocamento_y + altura2,
        deslocamento_x:deslocamento_x + largura2
    ] = imagem2

    return mosaico


def main():
    imagem1 = cv2.imread(IMAGEM_1)
    imagem2 = cv2.imread(IMAGEM_2)

    if imagem1 is None:
        raise FileNotFoundError(
            f"Não foi possível abrir {IMAGEM_1}. "
            "Verifique se está na mesma pasta do código."
        )

    if imagem2 is None:
        raise FileNotFoundError(
            f"Não foi possível abrir {IMAGEM_2}. "
            "Verifique se está na mesma pasta do código."
        )

    cinza1 = cv2.cvtColor(imagem1, cv2.COLOR_BGR2GRAY)
    cinza2 = cv2.cvtColor(imagem2, cv2.COLOR_BGR2GRAY)

    # PASSO 1 - SIFT
    sift = cv2.SIFT_create()

    pontos1, descritores1 = sift.detectAndCompute(cinza1, None)
    pontos2, descritores2 = sift.detectAndCompute(cinza2, None)

    print(f"Pontos SIFT na imagem 1: {len(pontos1)}")
    print(f"Pontos SIFT na imagem 2: {len(pontos2)}")

    if descritores1 is None or descritores2 is None:
        raise RuntimeError(
            "Não foi possível gerar descritores SIFT. "
            "Use imagens com mais detalhes visuais."
        )

    # PASSO 2 - MATCHING E TESTE DE LOWE
    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

    matches_brutos = matcher.knnMatch(
        descritores1,
        descritores2,
        k=2
    )

    bons_matches = []

    for par in matches_brutos:
        if len(par) < 2:
            continue

        primeiro, segundo = par

        if primeiro.distance < LIMIAR_LOWE * segundo.distance:
            bons_matches.append(primeiro)

    print(f"Matches após o teste de Lowe: {len(bons_matches)}")

    if len(bons_matches) < MINIMO_MATCHES:
        raise RuntimeError(
            f"Foram encontrados apenas {len(bons_matches)} bons matches. "
            f"São necessários pelo menos {MINIMO_MATCHES}. "
            "Tire novas fotos com maior sobreposição e mais textura."
        )

    pontos_origem = np.float32([
        pontos1[m.queryIdx].pt for m in bons_matches
    ]).reshape(-1, 1, 2)

    pontos_destino = np.float32([
        pontos2[m.trainIdx].pt for m in bons_matches
    ]).reshape(-1, 1, 2)

    # PASSO 3 - HOMOGRAFIA POR MÍNIMOS QUADRADOS
    H_ls, _ = cv2.findHomography(
        pontos_origem,
        pontos_destino,
        method=0
    )

    if H_ls is None:
        raise RuntimeError(
            "Não foi possível calcular a homografia por mínimos quadrados."
        )

    H_ls = normalizar_homografia(H_ls)

    # PASSO 4 - HOMOGRAFIA POR RANSAC
    H_ransac, mascara_ransac = cv2.findHomography(
        pontos_origem,
        pontos_destino,
        cv2.RANSAC,
        LIMIAR_REPROJECAO
    )

    if H_ransac is None or mascara_ransac is None:
        raise RuntimeError(
            "Não foi possível calcular a homografia com RANSAC."
        )

    H_ransac = normalizar_homografia(H_ransac)
    mascara_ransac = mascara_ransac.ravel().astype(bool)

    inliers = int(np.count_nonzero(mascara_ransac))
    outliers = int(len(bons_matches) - inliers)
    percentual_outliers = 100.0 * outliers / len(bons_matches)

    # RESULTADOS NUMÉRICOS
    np.set_printoptions(precision=6, suppress=True)

    print("\n--- Matriz de Homografia por Mínimos Quadrados ---")
    print(H_ls)

    print("\n--- Matriz de Homografia por RANSAC ---")
    print(H_ransac)

    print("\n--- Resultado do RANSAC ---")
    print(f"Total de bons matches: {len(bons_matches)}")
    print(f"Inliers: {inliers}")
    print(f"Outliers: {outliers}")
    print(f"Porcentagem de outliers: {percentual_outliers:.2f}%")

    diferenca_horizontal = abs(H_ls[0, 2] - H_ransac[0, 2])
    diferenca_vertical = abs(H_ls[1, 2] - H_ransac[1, 2])

    print("\n--- Comparação das translações ---")
    print(f"H_ls[0,2]: {H_ls[0, 2]:.6f}")
    print(f"H_ransac[0,2]: {H_ransac[0, 2]:.6f}")
    print(f"Diferença horizontal: {diferenca_horizontal:.6f}")

    print(f"H_ls[1,2]: {H_ls[1, 2]:.6f}")
    print(f"H_ransac[1,2]: {H_ransac[1, 2]:.6f}")
    print(f"Diferença vertical: {diferenca_vertical:.6f}")

    # PASSO 5 - CORRESPONDÊNCIAS
    imagem_matches = cv2.drawMatches(
        imagem1,
        pontos1,
        imagem2,
        pontos2,
        bons_matches,
        None,
        matchColor=(0, 255, 0),
        singlePointColor=None,
        matchesMask=mascara_ransac.astype(np.uint8).tolist(),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    cv2.imwrite(
        "correspondencias_ransac.jpg",
        imagem_matches
    )

    # PASSO 6 - MOSAICOS
    mosaico_ls = criar_mosaico(
        imagem1,
        imagem2,
        H_ls
    )

    mosaico_ransac = criar_mosaico(
        imagem1,
        imagem2,
        H_ransac
    )

    cv2.imwrite(
        "mosaico_minimos_quadrados.jpg",
        mosaico_ls
    )

    cv2.imwrite(
        "mosaico_ransac.jpg",
        mosaico_ransac
    )

    # EXIBIÇÃO
    figura, eixos = plt.subplots(3, 1, figsize=(14, 18))

    eixos[0].imshow(
        cv2.cvtColor(imagem_matches, cv2.COLOR_BGR2RGB)
    )
    eixos[0].set_title(
        "Correspondências aceitas pelo RANSAC"
    )
    eixos[0].axis("off")

    eixos[1].imshow(
        cv2.cvtColor(mosaico_ls, cv2.COLOR_BGR2RGB)
    )
    eixos[1].set_title(
        "Mosaico por Mínimos Quadrados Tradicionais"
    )
    eixos[1].axis("off")

    eixos[2].imshow(
        cv2.cvtColor(mosaico_ransac, cv2.COLOR_BGR2RGB)
    )
    eixos[2].set_title(
        "Mosaico com Homografia Robusta por RANSAC"
    )
    eixos[2].axis("off")

    plt.tight_layout()
    plt.show()

    print("\nArquivos gerados com sucesso:")
    print("- correspondencias_ransac.jpg")
    print("- mosaico_minimos_quadrados.jpg")
    print("- mosaico_ransac.jpg")


if __name__ == "__main__":
    main()



# Apêndice C — Código do processamento em tempo real

Arquivo correspondente: [`codigos/Lab3B_tempo_real.py`](codigos/Lab3B_tempo_real.py)


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# ============================================================
# LAB 3 - PARTE B
# DUAS WEBCAMS + SIFT + MATCHING + RANSAC + MOSAICO EM TEMPO REAL
#
# Teclas:
#   Q -> sair
#   S -> salvar as imagens e os resultados atuais
#
# Caso uma câmera não abra, altere:
#   CAMERA_1 = 0
#   CAMERA_2 = 1
# ============================================================

CAMERA_1 = 0
CAMERA_2 = 1

LARGURA = 640
ALTURA = 480

# Processa SIFT a cada N quadros para reduzir travamentos.
PROCESSAR_A_CADA = 3

# Parâmetros
LIMIAR_LOWE = 0.75
LIMIAR_RANSAC = 5.0
MINIMO_MATCHES = 4

# Limites para impedir mosaicos absurdamente grandes.
MAX_LARGURA_MOSAICO = 3000
MAX_ALTURA_MOSAICO = 2000


def abrir_camera(indice):
    """
    Abre uma webcam pelo índice informado.
    """
    cap = cv2.VideoCapture(indice)

    if cap.isOpened():
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, LARGURA)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTURA)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    return cap


def redimensionar_por_largura(imagem, largura_alvo):
    """
    Redimensiona uma imagem mantendo a proporção.
    """
    altura, largura = imagem.shape[:2]

    if largura == largura_alvo:
        return imagem

    escala = largura_alvo / largura
    nova_altura = max(1, int(altura * escala))

    return cv2.resize(
        imagem,
        (largura_alvo, nova_altura),
        interpolation=cv2.INTER_AREA
    )


def redimensionar_por_altura(imagem, altura_alvo):
    """
    Redimensiona uma imagem mantendo a proporção.
    """
    altura, largura = imagem.shape[:2]

    if altura == altura_alvo:
        return imagem

    escala = altura_alvo / altura
    nova_largura = max(1, int(largura * escala))

    return cv2.resize(
        imagem,
        (nova_largura, altura_alvo),
        interpolation=cv2.INTER_AREA
    )


def escrever_texto(imagem, texto, posicao=(10, 30),
                   cor=(255, 255, 255), escala=0.65):
    """
    Escreve texto com contorno para facilitar a leitura.
    """
    resultado = imagem.copy()

    cv2.putText(
        resultado,
        texto,
        posicao,
        cv2.FONT_HERSHEY_SIMPLEX,
        escala,
        (0, 0, 0),
        4,
        cv2.LINE_AA
    )

    cv2.putText(
        resultado,
        texto,
        posicao,
        cv2.FONT_HERSHEY_SIMPLEX,
        escala,
        cor,
        2,
        cv2.LINE_AA
    )

    return resultado


def tela_duas_cameras(frame1, frame2, mensagem):
    """
    Mostra as duas câmeras lado a lado.
    É usada quando ainda não existe uma homografia válida.
    """
    altura_comum = min(frame1.shape[0], frame2.shape[0])

    lado1 = redimensionar_por_altura(frame1, altura_comum)
    lado2 = redimensionar_por_altura(frame2, altura_comum)

    tela = np.hstack((lado1, lado2))

    tela = escrever_texto(
        tela,
        mensagem,
        posicao=(10, 30),
        cor=(0, 255, 255),
        escala=0.65
    )

    tela = escrever_texto(
        tela,
        "Q: sair | S: salvar",
        posicao=(10, tela.shape[0] - 15),
        cor=(255, 255, 255),
        escala=0.6
    )

    return tela


def criar_mosaico(imagem1, imagem2, H):
    """
    Projeta a imagem1 no plano da imagem2 usando a homografia H.
    """
    altura1, largura1 = imagem1.shape[:2]
    altura2, largura2 = imagem2.shape[:2]

    cantos1 = np.float32([
        [0, 0],
        [largura1, 0],
        [largura1, altura1],
        [0, altura1]
    ]).reshape(-1, 1, 2)

    cantos2 = np.float32([
        [0, 0],
        [largura2, 0],
        [largura2, altura2],
        [0, altura2]
    ]).reshape(-1, 1, 2)

    cantos1_transformados = cv2.perspectiveTransform(
        cantos1,
        H
    )

    todos_cantos = np.concatenate(
        (cantos1_transformados, cantos2),
        axis=0
    )

    x_min, y_min = np.floor(
        todos_cantos.min(axis=0).ravel()
    ).astype(int)

    x_max, y_max = np.ceil(
        todos_cantos.max(axis=0).ravel()
    ).astype(int)

    largura_canvas = x_max - x_min
    altura_canvas = y_max - y_min

    if largura_canvas <= 0 or altura_canvas <= 0:
        raise ValueError("Canvas inválido.")

    if (
        largura_canvas > MAX_LARGURA_MOSAICO
        or altura_canvas > MAX_ALTURA_MOSAICO
    ):
        raise ValueError("Homografia instável: mosaico muito grande.")

    deslocamento_x = -x_min
    deslocamento_y = -y_min

    matriz_deslocamento = np.array([
        [1.0, 0.0, deslocamento_x],
        [0.0, 1.0, deslocamento_y],
        [0.0, 0.0, 1.0]
    ], dtype=np.float64)

    mosaico = cv2.warpPerspective(
        imagem1,
        matriz_deslocamento @ H,
        (largura_canvas, altura_canvas)
    )

    x1 = deslocamento_x
    y1 = deslocamento_y
    x2 = x1 + largura2
    y2 = y1 + altura2

    if (
        x1 < 0
        or y1 < 0
        or x2 > mosaico.shape[1]
        or y2 > mosaico.shape[0]
    ):
        raise ValueError("Imagem 2 ficou fora do canvas.")

    mosaico[y1:y2, x1:x2] = imagem2

    return mosaico


def montar_tela_resultado(imagem_matches, mosaico, status):
    """
    Coloca as correspondências na parte superior
    e o mosaico na parte inferior.
    """
    largura_final = max(
        imagem_matches.shape[1],
        mosaico.shape[1]
    )

    matches_ajustado = redimensionar_por_largura(
        imagem_matches,
        largura_final
    )

    mosaico_ajustado = redimensionar_por_largura(
        mosaico,
        largura_final
    )

    matches_ajustado = escrever_texto(
        matches_ajustado,
        "Correspondencias SIFT aceitas pelo RANSAC",
        posicao=(10, 30),
        cor=(0, 255, 0),
        escala=0.65
    )

    mosaico_ajustado = escrever_texto(
        mosaico_ajustado,
        "Mosaico em tempo real",
        posicao=(10, 30),
        cor=(0, 255, 255),
        escala=0.65
    )

    tela = np.vstack(
        (matches_ajustado, mosaico_ajustado)
    )

    tela = escrever_texto(
        tela,
        status,
        posicao=(10, tela.shape[0] - 15),
        cor=(255, 255, 255),
        escala=0.58
    )

    return tela


def salvar_resultados(frame1, frame2, tela,
                      imagem_matches=None, mosaico=None):
    """
    Salva os resultados atuais em uma nova pasta.
    """
    horario = datetime.now().strftime("%Y%m%d_%H%M%S")

    pasta = Path(
        f"resultado_lab3_tempo_real_{horario}"
    )

    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

    cv2.imwrite(
        str(pasta / "camera1.jpg"),
        frame1
    )

    cv2.imwrite(
        str(pasta / "camera2.jpg"),
        frame2
    )

    cv2.imwrite(
        str(pasta / "tela_completa.jpg"),
        tela
    )

    if imagem_matches is not None:
        cv2.imwrite(
            str(pasta / "correspondencias.jpg"),
            imagem_matches
        )

    if mosaico is not None:
        cv2.imwrite(
            str(pasta / "mosaico.jpg"),
            mosaico
        )

    print(f"Resultados salvos em: {pasta.resolve()}")


def main():
    cap1 = abrir_camera(CAMERA_1)
    cap2 = abrir_camera(CAMERA_2)

    if not cap1.isOpened():
        print(f"Erro: câmera {CAMERA_1} não abriu.")

    if not cap2.isOpened():
        print(f"Erro: câmera {CAMERA_2} não abriu.")

    if not cap1.isOpened() or not cap2.isOpened():
        cap1.release()
        cap2.release()
        cv2.destroyAllWindows()

        print("Altere CAMERA_1 e CAMERA_2 no início do código.")
        return

    try:
        sift = cv2.SIFT_create()
    except AttributeError:
        print("Erro: sua instalação do OpenCV não possui SIFT.")
        print("Instale com: python3 -m pip install opencv-contrib-python")
        cap1.release()
        cap2.release()
        return

    matcher = cv2.BFMatcher(
        cv2.NORM_L2,
        crossCheck=False
    )

    contador = 0

    tela_atual = None
    frame1_atual = None
    frame2_atual = None
    matches_atual = None
    mosaico_atual = None

    print("Duas câmeras abertas com sucesso.")
    print("Q: sair")
    print("S: salvar resultados")

    while True:
        sucesso1, frame1 = cap1.read()
        sucesso2, frame2 = cap2.read()

        if not sucesso1 or not sucesso2:
            print("Erro ao ler uma das câmeras.")
            break

        frame1_atual = frame1.copy()
        frame2_atual = frame2.copy()

        contador += 1

        if contador % PROCESSAR_A_CADA == 0 or tela_atual is None:
            cinza1 = cv2.cvtColor(
                frame1,
                cv2.COLOR_BGR2GRAY
            )

            cinza2 = cv2.cvtColor(
                frame2,
                cv2.COLOR_BGR2GRAY
            )

            pontos1, descritores1 = sift.detectAndCompute(
                cinza1,
                None
            )

            pontos2, descritores2 = sift.detectAndCompute(
                cinza2,
                None
            )

            status = (
                f"SIFT1: {len(pontos1)} | "
                f"SIFT2: {len(pontos2)}"
            )

            if descritores1 is None or descritores2 is None:
                tela_atual = tela_duas_cameras(
                    frame1,
                    frame2,
                    status + " | Sem descritores suficientes"
                )

                matches_atual = None
                mosaico_atual = None

            else:
                pares = matcher.knnMatch(
                    descritores1,
                    descritores2,
                    k=2
                )

                bons_matches = []

                for par in pares:
                    if len(par) < 2:
                        continue

                    primeiro, segundo = par

                    if (
                        primeiro.distance
                        < LIMIAR_LOWE * segundo.distance
                    ):
                        bons_matches.append(primeiro)

                status += f" | Matches: {len(bons_matches)}"

                if len(bons_matches) < MINIMO_MATCHES:
                    tela_atual = tela_duas_cameras(
                        frame1,
                        frame2,
                        status + " | Poucos matches"
                    )

                    matches_atual = None
                    mosaico_atual = None

                else:
                    origem = np.float32([
                        pontos1[m.queryIdx].pt
                        for m in bons_matches
                    ]).reshape(-1, 1, 2)

                    destino = np.float32([
                        pontos2[m.trainIdx].pt
                        for m in bons_matches
                    ]).reshape(-1, 1, 2)

                    H, mascara = cv2.findHomography(
                        origem,
                        destino,
                        cv2.RANSAC,
                        LIMIAR_RANSAC
                    )

                    if H is None or mascara is None:
                        tela_atual = tela_duas_cameras(
                            frame1,
                            frame2,
                            status + " | Homografia não encontrada"
                        )

                        matches_atual = None
                        mosaico_atual = None

                    else:
                        mascara = mascara.ravel().astype(np.uint8)

                        inliers = int(
                            np.count_nonzero(mascara)
                        )

                        outliers = int(
                            len(bons_matches) - inliers
                        )

                        status += (
                            f" | Inliers: {inliers}"
                            f" | Outliers: {outliers}"
                        )

                        imagem_matches = cv2.drawMatches(
                            frame1,
                            pontos1,
                            frame2,
                            pontos2,
                            bons_matches,
                            None,
                            matchColor=(0, 255, 0),
                            singlePointColor=None,
                            matchesMask=mascara.tolist(),
                            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
                        )

                        try:
                            mosaico = criar_mosaico(
                                frame1,
                                frame2,
                                H
                            )

                            tela_atual = montar_tela_resultado(
                                imagem_matches,
                                mosaico,
                                status + " | Q: sair | S: salvar"
                            )

                            matches_atual = imagem_matches.copy()
                            mosaico_atual = mosaico.copy()

                        except ValueError as erro:
                            tela_atual = tela_duas_cameras(
                                frame1,
                                frame2,
                                status + f" | {erro}"
                            )

                            matches_atual = imagem_matches.copy()
                            mosaico_atual = None

        tela_exibicao = tela_atual

        altura_maxima = 900

        if tela_exibicao.shape[0] > altura_maxima:
            tela_exibicao = redimensionar_por_altura(
                tela_exibicao,
                altura_maxima
            )

        cv2.imshow(
            "Lab 3 - Parte B - Tempo Real",
            tela_exibicao
        )

        tecla = cv2.waitKey(1) & 0xFF

        if tecla == ord("q"):
            print("Programa encerrado.")
            break

        if tecla == ord("s"):
            salvar_resultados(
                frame1_atual,
                frame2_atual,
                tela_atual,
                matches_atual,
                mosaico_atual
            )

    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
